# Lab | Simple LLM App with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

### LangSmith

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls.
As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent.
The best way to do this is with [LangSmith](https://smith.langchain.com).

After you sign up at the link above, make sure to set your environment variables to start logging traces:

```shell
export LANGCHAIN_TRACING_V2="true"
export LANGCHAIN_API_KEY="..."
```

Or, if in a notebook, you can set them with:

```python
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
```

In [ ]:
# Listing all available models from GROQ
import requests
import os
import json
from dotenv import load_dotenv, find_dotenv

# Loading .env API keys
_ = load_dotenv(find_dotenv())

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(json.dumps(response.json(), indent=2))

In [2]:
# Choosing a model
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

Let's first use the model directly. `ChatModel`s are instances of LangChain "Runnables", which means they expose a standard interface for interacting with them. To just simply call the model, we can pass in a list of messages to the `.invoke` method.

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English into Portuguese"), #Feel free to try something else here
    HumanMessage(content="Today is a sunny day. I went to the beach and get a tan"),
]

model.invoke(messages)

AIMessage(content='Hoje é um dia ensolarado. Fui à praia e peguei um bronzeado.\n\nNote: A more literal translation of "get a tan" would be "obter um bronzeado" or "ficar bronzeado", but "pegar um bronzeado" is a common and idiomatic way to express the idea in Portuguese.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 57, 'total_tokens': 130, 'completion_time': 0.236339454, 'completion_tokens_details': None, 'prompt_time': 0.005718128, 'prompt_tokens_details': None, 'queue_time': 0.09250895, 'total_time': 0.242057582}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019cb396-22fc-7381-9999-2eda57fccbbb-0', usage_metadata={'input_tokens': 57, 'output_tokens': 73, 'total_tokens': 130})

If we've enable LangSmith, we can see that this run is logged to LangSmith, and can see the [LangSmith trace](https://smith.langchain.com/public/88baa0b2-7c1a-4d09-ba30-a47985dde2ea/r)

## OutputParsers

Notice that the response from the model is an `AIMessage`. This contains a string response along with other metadata about the response. Oftentimes we may just want to work with the string response. We can parse out just this response by using a simple output parser.

We first import the simple output parser.

In [4]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

One way to use it is to use it by itself. For example, we could save the result of the language model call and then pass it to the parser.

In [5]:
result = model.invoke(messages)

In [6]:
parser.invoke(result)

'Hoje é um dia ensolarado. Fui à praia e peguei um bronzeado.\n\nNote: In Portuguese, "get a tan" is more commonly translated to "pegar um bronzeado" (to get a tan) or simply "bronzeado" (tan). Also, "ensolarado" is a more common adjective to describe a sunny day in Portuguese, rather than "sunny" which is "ensolarado" or "soalheiro" in some regions of Brazil and Portugal.'

More commonly, we can "chain" the model with this output parser. This means this output parser will get called everytime in this chain. This chain takes on the input type of the language model (string or list of message) and returns the output type of the output parser (string).

We can easily create the chain using the `|` operator. The `|` operator is used in LangChain to combine two elements together.

In [7]:
chain = model | parser

In [8]:
chain.invoke(messages)

'Hoje é um dia ensolarado. Eu fui à praia e peguei um bronzeado.\n\nNote: In Portuguese, "get a tan" is translated to "pegar um bronzeado", which is a common expression in Brazil. In some European Portuguese countries, you might use "tomar sol" (to sunbathe) or "ficar moreno" (to get tanned) instead.'

If we now look at LangSmith, we can see that the chain has two steps: first the language model is called, then the result of that is passed to the output parser. We can see the [LangSmith trace]( https://smith.langchain.com/public/f1bdf656-2739-42f7-ac7f-0f1dd712322f/r)

## Prompt Templates

Right now we are passing a list of messages directly into the language model. Where does this list of messages come from? Usually, it is constructed from a combination of user input and application logic. This application logic usually takes the raw user input and transforms it into a list of messages ready to pass to the language model. Common transformations include adding a system message or formatting a template with the user input.

PromptTemplates are a concept in LangChain designed to assist with this transformation. They take in raw user input and return data (a prompt) that is ready to pass into a language model. 

Let's create a PromptTemplate here. It will take in two user variables:

- `language`: The language to translate text into
- `text`: The text to translate

In [9]:
from langchain_core.prompts import ChatPromptTemplate

First, let's create a string that we will format to be the system message:

In [10]:
system_template = "Translate the following into {language}:"

Next, we can create the PromptTemplate. This will be a combination of the `system_template` as well as a simpler template for where the put the text

In [11]:
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

The input to this prompt template is a dictionary. We can play around with this prompt template by itself to see what it does by itself

In [12]:
result = prompt_template.invoke({"language": "portuguese", "text": "I'm swamped by work. I hope the boss accepts my vacation proposal."})

result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into portuguese:', additional_kwargs={}, response_metadata={}), HumanMessage(content="I'm swamped by work. I hope the boss accepts my vacation proposal.", additional_kwargs={}, response_metadata={})])

We can see that it returns a `ChatPromptValue` that consists of two messages. If we want to access the messages directly we do:

In [13]:
result.to_messages()

[SystemMessage(content='Translate the following into portuguese:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I'm swamped by work. I hope the boss accepts my vacation proposal.", additional_kwargs={}, response_metadata={})]

## Chaining together components with LCEL

We can now combine this with the model and the output parser from above using the pipe (`|`) operator:

In [14]:
chain = prompt_template | model | parser

In [15]:
chain.invoke({"language": "portuguese", "text": "I'm swamped by work. I hope the boss accepts my vacation proposal."})

'Estou afogado em trabalho. Espero que o chefe aceite minha proposta de férias.\n\nor, in a more natural Portuguese translation:\n\nEstou sobrecarregado de trabalho. Espero que o chefe aceite meu pedido de férias.\n\nNote: "Sobrecarregado" is a more common way to say "swamped" in Portuguese, and "pedido de férias" is a more natural way to say "vacation proposal".'

This is a simple example of using [LangChain Expression Language (LCEL)](/docs/concepts/#langchain-expression-language-lcel) to chain together LangChain modules. There are several benefits to this approach, including optimized streaming and tracing support.

If we take a look at the LangSmith trace, we can see all three components show up in the [LangSmith trace](https://smith.langchain.com/public/bc49bec0-6b13-4726-967f-dbd3448b786d/r).

## Serving with LangServe

Now that we've built an application, we need to serve it. That's where LangServe comes in.
LangServe helps developers deploy LangChain chains as a REST API. You do not need to use LangServe to use LangChain, but in this guide we'll show how you can deploy your app with LangServe.

While the first part of this guide was intended to be run in a Jupyter Notebook or script, we will now move out of that. We will be creating a Python file and then interacting with it from the command line.

Install with:
```bash
pip install "langserve[all]"
```

### Server

To create a server for our application we'll make a `serve.py` file. This will contain our logic for serving our application. It consists of three things:
1. The definition of our chain that we just built above
2. Our FastAPI app
3. A definition of a route from which to serve the chain, which is done with `langserve.add_routes`


```python
#!/usr/bin/env python
from typing import List

from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langserve import add_routes

# 1. Create prompt template
system_template = "Translate the following into {language}:"
prompt_template = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('user', '{text}')
])

# 2. Create model
model = ChatOpenAI()

# 3. Create parser
parser = StrOutputParser()

# 4. Create chain
chain = prompt_template | model | parser


# 4. App definition
app = FastAPI(
  title="LangChain Server",
  version="1.0",
  description="A simple API server using LangChain's Runnable interfaces",
)

# 5. Adding chain route

add_routes(
    app,
    chain,
    path="/chain",
)

if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="localhost", port=8000)
```

And that's it! If we execute this file:
```bash
python serve.py
```
we should see our chain being served at [http://localhost:8000](http://localhost:8000).

### Playground

Every LangServe service comes with a simple [built-in UI](https://github.com/langchain-ai/langserve/blob/main/README.md#playground) for configuring and invoking the application with streaming output and visibility into intermediate steps.
Head to [http://localhost:8000/chain/playground/](http://localhost:8000/chain/playground/) to try it out! Pass in the same inputs as before - `{"language": "italian", "text": "hi"}` - and it should respond same as before.

### Client

Now let's set up a client for programmatically interacting with our service. We can easily do this with the `[langserve.RemoteRunnable](/docs/langserve/#client)`.
Using this, we can interact with the served chain as if it were running client-side.

In [16]:
from langserve import RemoteRunnable

remote_chain = RemoteRunnable("http://localhost:8000/chain/")
remote_chain.invoke({"language": "japanese", "text": "I'm feeling tired today. I need to take a nap."})

"(Kyō wa tsukarete imasu. Hirune ga hitsuyō desu.)\n\nHere's a breakdown of the translation:\n\n* (Kyō wa) - Today\n* (tsukarete imasu) - I'm feeling tired\n* (Hirune ga hitsuyō desu) - I need to take a nap\n\nAlternatively, you could also say:\n\n(Kyō wa nemui desu. Nemutai to omoimasu.)\n\nThis translation is a bit more casual and conversational:\n\n* (Kyō wa) - Today\n* (nemui desu) - I'm feeling sleepy\n* (Nemutai to omoimasu) - I think I'll take a nap"

To learn more about the many other features of LangServe [head here](/docs/langserve).

## Conclusion

That's it! In this tutorial you've learned how to create your first simple LLM application. You've learned how to work with language models, how to parse their outputs, how to create a prompt template, chaining them with LCEL, how to get great observability into chains you create with LangSmith, and how to deploy them with LangServe.

This just scratches the surface of what you will want to learn to become a proficient AI Engineer. Luckily - we've got a lot of other resources!

For further reading on the core concepts of LangChain, we've got detailed [Conceptual Guides](/docs/concepts).

If you have more specific questions on these concepts, check out the following sections of the how-to guides:

- [LangChain Expression Language (LCEL)](/docs/how_to/#langchain-expression-language-lcel)
- [Prompt templates](/docs/how_to/#prompt-templates)
- [Chat models](/docs/how_to/#chat-models)
- [Output parsers](/docs/how_to/#output-parsers)
- [LangServe](/docs/langserve/)

And the LangSmith docs:

- [LangSmith](https://docs.smith.langchain.com)

# Key Takeways

## What We Built

We built **a multilingual translation app powered by an open-source LLM, served as a REST API**. The system has two main technical components:

- **LCEL Chain** (`prompt_template | model | parser`): combines a `ChatPromptTemplate`, a `ChatGroq` model, and a `StrOutputParser` into a single composable pipeline that takes `{language}` and `{text}` as inputs and returns a translated string
- **LangServe API** (`serve.py`): wraps the chain in a FastAPI server, exposing it at `/chain/` with a built-in Playground UI and a `RemoteRunnable` client interface

Together, these enable end-to-end translation from a notebook client through a locally hosted REST API, demonstrating a production-ready LLM app architecture.

---

## Extensions We Implemented

### **Groq Integration**
Replaced the default `ChatOpenAI` model with `ChatGroq(model="llama-3.3-70b-versatile")`, using Meta's open-source Llama 3.3 70B model via Groq's fast inference API. This avoids OpenAI dependency and uses a fully open-source model with a 131K token context window.

### **Dynamic Model Discovery**
Added a cell that queries the Groq API (`/openai/v1/models`) and pretty-prints all available models with `json.dumps(..., indent=2)`, allowing informed model selection rather than hardcoding a name blindly.

### **Environment Management with dotenv**
Configured `load_dotenv()` with `find_dotenv()` across both the notebook and `serve.py` to securely load `GROQ_API_KEY` and LangSmith credentials from a `.env` file, keeping secrets out of the codebase.

---

## What We Learned

| Topic | Key Insight |
|-------|-------------|
| **LCEL Pipe Operator** | The `\|` operator chains Runnables (`prompt_template \| model \| parser`) so each component's output becomes the next one's input, with no boilerplate glue code |
| **AIMessage vs String** | `model.invoke()` returns an `AIMessage` object with metadata; `StrOutputParser` extracts just the string content, which is what downstream components and users typically need |
| **ChatPromptTemplate** | Templates decouple prompt structure from user input — variables like `{language}` and `{text}` are injected at runtime, making the chain reusable across any language pair |
| **LangServe Architecture** | Wrapping a chain in FastAPI via `add_routes` exposes it as a REST API; the client (`RemoteRunnable`) interacts with it identically to a local chain, abstracting away HTTP entirely |
| **Client-Server Separation** | In a deployed scenario, API keys and model logic stay on the server — clients only send inputs and receive outputs, improving security and maintainability |
| **Groq Model Selection** | Not all models listed in the Groq API are suitable for chat/translation — audio models (Whisper), guard models, and TTS models must be filtered out; `llama-3.3-70b-versatile` offers the best quality for language tasks |
| **dotenv Scope** | `load_dotenv()` must be called before any LangChain component is initialised; env vars set after import are not picked up by already-instantiated objects |